In [ ]:
import os
import numpy as np
import rasterio
from rasterio.warp import reproject
from rasterio.enums import Resampling as ResampleEnums
from skimage.morphology import remove_small_objects, remove_small_holes, opening, rectangle, dilation, square
from skimage.measure import label, regionprops
from sklearn.cluster import MiniBatchKMeans
import matplotlib.pyplot as plt

In [ ]:
### Loading and establishing filepaths

# 1. Reference image
ref_image_path = "C:/Users/mon/Downloads/19_clipped.tif"

# 2. Output directory
ndvi_output_dir = "C:/Users/mon/Downloads/fullseason3_jupy_ndvi/"
os.makedirs(ndvi_output_dir, exist_ok=True)

# 3. Sentinel-2 folders
date_folders = {
    "20190403": "C:/Users/mon/Downloads/2019 Sentinel-2 Bands/4 - April/S2A_17TLJ_20190415_1_L2A",
    "20190508": "C:/Users/mon/Downloads/2019 Sentinel-2 Bands/5 - May/S2A_17TLJ_20190505_0_L2A",
    "20190627": "C:/Users/mon/Downloads/2019 Sentinel-2 Bands/6 - June/S2A_17TLJ_20190604_0_L2A",
    "20190714": "C:/Users/mon/Downloads/2019 Sentinel-2 Bands/7 - July/S2A_17TLJ_20190714_L2A",
    "20190801": "C:/Users/mon/Downloads/2019 Sentinel-2 Bands/8 - August/S2A_17TLJ_20190803_0_L2A"
}
B04 = "B04.tif"
B08 = "B08.tif"

# 4. Load reference image
with rasterio.open(ref_image_path) as ref:
    ref_crs = ref.crs
    ref_transform = ref.transform
    ref_width = ref.width
    ref_height = ref.height
    ref_profile = ref.profile.copy()

In [ ]:
### Functions - align_band and compute_ndvi

# 5. Align bands
def align_band(input_path, output_path):
    with rasterio.open(input_path) as src:
        meta = src.meta.copy()
        meta.update({
            "crs": ref_crs,
            "transform": ref_transform,
            "width": ref_width,
            "height": ref_height
        })
        with rasterio.open(output_path, "w", **meta) as dst:
            reproject(
                source=rasterio.band(src, 1),
                destination=rasterio.band(dst, 1),
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=ref_transform,
                dst_crs=ref_crs,
                resampling=ResampleEnums.bilinear
            )

# 6. NDVI computation
def compute_ndvi(red_path, nir_path, out_path):
    with rasterio.open(red_path) as red_src, rasterio.open(nir_path) as nir_src:
        red = red_src.read(1).astype(np.float32)
        nir = nir_src.read(1).astype(np.float32)
        ndvi = (nir - red) / (nir + red + 1e-10)
        ndvi[np.isnan(ndvi)] = 0
        profile = red_src.profile.copy()
        profile.update(dtype=rasterio.float32, count=1)
        with rasterio.open(out_path, "w", **profile) as dst:
            dst.write(ndvi, 1)


In [ ]:
# 7. Align red/NIR bands
for date in sorted(date_folders.keys()):
    folder = date_folders[date]
    red_input = os.path.join(folder, B04)
    nir_input = os.path.join(folder, B08)
    red_aligned = os.path.join(ndvi_output_dir, f"aligned_B04_{date}.tif")
    nir_aligned = os.path.join(ndvi_output_dir, f"aligned_B08_{date}.tif")
    align_band(red_input, red_aligned)
    align_band(nir_input, nir_aligned)

In [ ]:
# 8. NDVI stack
ndvi_stack = []
for date in sorted(date_folders.keys()):
    red_aligned = os.path.join(ndvi_output_dir, f"aligned_B04_{date}.tif")
    nir_aligned = os.path.join(ndvi_output_dir, f"aligned_B08_{date}.tif")
    ndvi_output = os.path.join(ndvi_output_dir, f"ndvi_{date}.tif")
    compute_ndvi(red_aligned, nir_aligned, ndvi_output)
    with rasterio.open(ndvi_output) as src:
        ndvi_stack.append(src.read(1))
ndvi_stack = np.stack(ndvi_stack, axis=-1)

In [ ]:
# 9. KMeans clustering
H, W, T = ndvi_stack.shape
ndvi_flat = ndvi_stack.reshape(-1, T)
valid_mask = ~np.all(ndvi_flat == 0, axis=1)
ndvi_valid = ndvi_flat[valid_mask]

kmeans = MiniBatchKMeans(n_clusters=14, random_state=42, batch_size=10000).fit(ndvi_valid)
cluster_labels = np.full(ndvi_flat.shape[0], -1, dtype=np.int32)
cluster_labels[valid_mask] = kmeans.labels_
cluster_labels = cluster_labels.reshape(H, W)


In [ ]:
### Printing cluster stats and assigned clusters

# 10. Crop NDVI shape-based reference
crop_refs = {
    "corn": {"peak": (0.60, 0.96), "std": (0.08, 0.45), "min": (0.04, 0.40)},
    "winterwheat": {"peak": (0.50, 0.90), "std": (0.08, 0.40), "min": (0.05, 0.50)},
    "soybeans": {"peak": (0.58, 0.92), "std": (0.06, 0.40), "min": (0.04, 0.45)},
    "alfalfa": {"peak": (0.60, 0.85), "std": (0.10, 0.35), "min": (0.08, 0.40)},
    "drybeans": {"peak": (0.55, 0.90), "std": (0.06, 0.40), "min": (0.04, 0.42)},
    "sugarbeets": {"peak": (0.60, 0.95), "std": (0.08, 0.40), "min": (0.05, 0.40)}
}

crop_to_clusters = {c: [] for c in crop_refs}

print("\nCluster NDVI Stats:")
for i in range(14):
    cluster_ndvi = ndvi_valid[kmeans.labels_ == i]
    if cluster_ndvi.size == 0:
        continue
    avg_curve = cluster_ndvi.mean(axis=0)
    peak = float(np.max(avg_curve))
    std_dev = float(np.std(avg_curve))
    min_val = float(np.min(avg_curve))
    print(f"Cluster {i:2d} | Peak: {peak:.3f}, Std: {std_dev:.3f}, Min: {min_val:.3f}")
    for crop, ref in crop_refs.items():
        if ref["peak"][0] <= peak <= ref["peak"][1] and \
           ref["std"][0] <= std_dev <= ref["std"][1] and \
           ref["min"][0] <= min_val <= ref["min"][1]:
            crop_to_clusters[crop].append(i)

print("\nCluster → Crop Assignments:")
for crop, clist in crop_to_clusters.items():
    unique_clusters = sorted(set(clist))
    print(f"{crop:12s} → Clusters: {unique_clusters}")

In [ ]:
# 10B. Plot cluster profiles
dates_sorted = sorted(date_folders.keys())
plt.figure(figsize=(10, 6))
for i in range(14):
    cluster_ndvi = ndvi_valid[kmeans.labels_ == i]
    if cluster_ndvi.size == 0:
        continue
    avg_curve = cluster_ndvi.mean(axis=0)
    plt.plot(dates_sorted, avg_curve, label=f"Cluster {i}")
plt.title("NDVI Time Series for Each KMeans Cluster")
plt.xlabel("Date")
plt.ylabel("NDVI")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(ndvi_output_dir, "cluster_profiles_k18.png"), dpi=150)
plt.show()

In [ ]:
# 11. Load SWIR + NDBI for filtering
swir_path = os.path.join(date_folders["20190714"], "B11.tif")
swir2_path = os.path.join(date_folders["20190714"], "B12.tif")
aligned_swir = os.path.join(ndvi_output_dir, "aligned_B11_20190714.tif")
aligned_swir2 = os.path.join(ndvi_output_dir, "aligned_B12_20190714.tif")
align_band(swir_path, aligned_swir)
align_band(swir2_path, aligned_swir2)

with rasterio.open(aligned_swir) as swir_src, rasterio.open(aligned_swir2) as swir2_src:
    swir = swir_src.read(1).astype(np.float32)
    swir2 = swir2_src.read(1).astype(np.float32)

ndvi_july = ndvi_stack[:, :, sorted(date_folders.keys()).index("20190714")]
ndbi = (swir - ndvi_july) / (swir + ndvi_july + 1e-10)

In [ ]:
# 12. Apply crop masks
mask = np.zeros((H, W), dtype=bool)
for crop, clusters in crop_to_clusters.items():
    cluster_mask = np.isin(cluster_labels, clusters)
    if crop in ["corn", "soybeans", "sugarbeets"]:
        sw1 = np.percentile(swir, 30)
        sw2 = np.percentile(swir2, 50)
        mask |= cluster_mask & (swir > sw1) & (swir2 > sw2) & (ndbi > 0.05)
    elif crop in ["winterwheat", "alfalfa", "drybeans"]:
        sw1 = np.percentile(swir, 20)
        sw2 = np.percentile(swir2, 40)
        mask |= cluster_mask & (swir > sw1) & (swir2 > sw2) & (ndbi > 0.03)

plt.figure(figsize=(8, 8))
plt.imshow(mask, cmap='gray')
plt.title("Initial Cropland Mask")
plt.axis('off')
plt.show()

In [ ]:
# 13. CDL road overlay
cdl_path = "C:/Users/mon/Downloads/2019cdl_matched2sentinel.tif"
with rasterio.open(cdl_path) as cdl_src:
    cdl_array = cdl_src.read(1)
road_mask = (cdl_array >= 121) & (cdl_array <= 124)
road_mask = dilation(road_mask, square(2))
road_mask = dilation(road_mask, square(2))
mask[road_mask] = 0  # Remove roads from cropland mask

In [ ]:
# 14. Morphological cleanup
mask = remove_small_objects(mask.astype(bool), min_size=2500)
mask = remove_small_holes(mask, area_threshold=3000)
mask = opening(mask.astype(np.uint8), rectangle(3, 10))
mask = opening(mask, rectangle(6, 10))

plt.figure(figsize=(8, 8))
plt.imshow(mask, cmap='gray')
plt.title("Morphologically-cleaned Binary Cropland Mask")
plt.axis('off')
plt.show()

In [ ]:
# 15. Aspect ratio filtering
labeled_mask = label(mask)
final_mask = np.zeros_like(mask, dtype=bool)
for region in regionprops(labeled_mask):
    if region.area < 3000:
        continue
    top, left, bottom, right = region.bbox
    height = bottom - top
    width = right - left
    if height == 0 or width == 0:
        continue
    aspect_ratio = max(height, width) / min(height, width)
    if aspect_ratio < 15.0:
        final_mask[labeled_mask == region.label] = True

plt.figure(figsize=(8, 8))
plt.imshow(final_mask, cmap='gray')
plt.title("Final Cropland Mask (After Aspect Ratio Filtering)")
plt.axis('off')
plt.show()

In [ ]:
### 16. Sentinel-2 and CDL RGB overlay for visual accuracy

# Extract the 6 main huron county crops from CDL and create mask
cdl_mask = np.isin(cdl_array, [1, 5, 9, 23, 24, 36])
crop_mask = mask.astype(bool)

# Rebuild overlay image
overlay = np.zeros((*crop_mask.shape, 3), dtype=np.uint8)
overlay[crop_mask & cdl_mask] = [0, 255, 0]     # True Positive → Green
overlay[~crop_mask & cdl_mask] = [255, 0, 0]    # False Negative → Red
overlay[crop_mask & ~cdl_mask] = [0, 0, 255]    # False Positive → Blue

# Calculate stats
tp = np.sum(crop_mask & cdl_mask)
fn = np.sum(~crop_mask & cdl_mask)
fp = np.sum(crop_mask & ~cdl_mask)
total = tp + fn + fp
combined_accuracy = tp / total * 100 if total > 0 else 0

# Create side-by-side plot
fig, axs = plt.subplots(1, 2, figsize=(16, 8))

# Show overlay image
axs[0].imshow(overlay)
axs[0].set_title("Overlay: Green = Match, Red = Missed, Blue = False Positive")
axs[0].axis('off')

# Show accuracy breakdown
labels = ['True Positives', 'False Negatives', 'False Positives']
values = [tp, fn, fp]
colors = ['green', 'red', 'blue']
percentages = [v / total * 100 for v in values] if total > 0 else [0, 0, 0]

bars = axs[1].bar(labels, percentages, color=colors)
axs[1].set_ylim(0, 100)
axs[1].set_ylabel("Percentage of Pixels (%)")
axs[1].set_title(f"Accuracy Breakdown\nCombined Accuracy: {combined_accuracy:.2f}%")

for bar, pct in zip(bars, percentages):
    axs[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f"{pct:.1f}%", ha='center', va='bottom', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# 17. Save final cropland mask
output_path = "C:/Users/mon/Downloads/fullseason3_cropland_mask.tif"
mask_profile = ref_profile.copy()
mask_profile.update(dtype=rasterio.uint8, count=1)

with rasterio.open(output_path, "w", **mask_profile) as dst:
    dst.write(final_mask.astype(np.uint8), 1)